## Modeling

In [1]:
from joblib import load
from pathlib import Path

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
RANDOM_STATE = 22
ARTIFACTS_DIR = Path("artifacts")
if not ARTIFACTS_DIR.is_dir():
    ARTIFACTS_DIR = Path("..") / ARTIFACTS_DIR

In [3]:
preprocessing_artifact = load(ARTIFACTS_DIR / "preprocessing.joblib")
preprocessor = preprocessing_artifact["preprocessor"]
target = preprocessing_artifact["target"]

if "train" in preprocessing_artifact and "test" in preprocessing_artifact:
    X_train = preprocessing_artifact["train"]
    X_test = preprocessing_artifact["test"]
else:
    X_train = pd.read_csv(ARTIFACTS_DIR / "X_train.csv", index_col=0)
    X_test = pd.read_csv(ARTIFACTS_DIR / "x_test.csv", index_col=0)

# Match labels to feature rows using the saved split indices.
y_train = pd.read_csv(ARTIFACTS_DIR / "y_train.csv", index_col=0)[target].reindex(X_train.index)
y_test = pd.read_csv(ARTIFACTS_DIR / "y_test.csv", index_col=0)[target].reindex(X_test.index)

if y_train.isna().any() or y_test.isna().any():
    raise ValueError("Missing labels for the saved splits. Rerun preprocessing.ipynb to regenerate the artifacts.")

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((1600, 15), (1600,), (400, 15), (400,))

In [5]:
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

In [6]:
param_spaces = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__class_weight": [None, "balanced"]
    },

    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10, 20],
        "classifier__min_samples_leaf": [1, 2, 4],
        "classifier__class_weight": [None, "balanced"]
    },

    "Gradient Boosting": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__learning_rate": [0.01, 0.05, 0.1],
        "classifier__max_depth": [2, 3, 5]
    },

    "SVM": {
        "classifier__C": [0.1, 1, 10, 100],
        "classifier__gamma": ["scale", 0.01, 0.1],
        "classifier__class_weight": [None, "balanced"]
    }
}


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "SVM": SVC(
        kernel="rbf",
        random_state=RANDOM_STATE
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision_macro",
            "recall": "recall_macro",
            "f1": "f1_macro"
        }
    )

    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1 Macro": scores["test_f1"].mean()
    })

results_df = pd.DataFrame(results)
results_df.sort_values("F1 Macro", ascending=False)


,Model,Accuracy,Precision,Recall,F1 Macro
0,Logistic Regression,0.760000,0.762180,0.759426,0.760078
3,SVM,0.741875,0.747090,0.741348,0.743101
2,Gradient Boosting,0.701875,0.702141,0.701156,0.700508
1,Random Forest,0.697500,0.694366,0.696591,0.694062
